# Notebook 03 — Filter, Merge & Clean

## Overview
Core data cleaning pipeline: filters both sources, merges them, enriches missing descriptions, and produces the final clean dataset.

**Output:** `Data/Clean/merged_non_western_fantasy.json`

## 1. Setup & Load

In [1]:
import json, re, time, os, requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from langdetect import detect
import yaml

load_dotenv()

def find_repo_root():
    current = Path().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "config.yaml").exists():
            return parent
    raise FileNotFoundError("config.yaml not found")

REPO_ROOT = find_repo_root()
with open(REPO_ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

RAW_DIR   = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"
CLEAN_DIR = REPO_ROOT / "Data" / "Clean"
CLEAN_DIR.mkdir(exist_ok=True)

OL_OUT     = REPO_ROOT / config["output_data_raw_non_western_fantasy"]["ol_results"]
GR_OUT     = REPO_ROOT / config["output_data_raw_non_western_fantasy"]["gr_results"]
MERGED_OUT = REPO_ROOT / config["output_data_clean_non_western_fantasy"]["merged"]

GOOGLE_BOOKS_API_KEY = os.getenv("GOOGLE_BOOKS_API_KEY", "")
GB_CKPT  = CLEAN_DIR / "google_books_checkpoint.json"
OL_CKPT  = CLEAN_DIR / "ol_desc_checkpoint.json"

ol_raw = pd.DataFrame(json.load(open(OL_OUT)))
gr_raw = pd.read_csv(GR_OUT)

print(f"OL books:        {len(ol_raw):,}")
print(f"Goodreads books: {len(gr_raw):,}")
print(f"Google Books key: {'yes' if GOOGLE_BOOKS_API_KEY else 'missing'}")

OL books:        4,357
Goodreads books: 2,129
Google Books key: yes


## 2. Filter Sets

In [2]:
WESTERN_NOISE_AUTHORS = {
    'edgar rice burroughs', 'h. rider haggard', 'rudyard kipling',
    'joseph conrad', 'wilbur smith', 'james michener', 'john buchan',
    'arthur conan doyle', 'h.g. wells', 'h. g. wells', 'jules verne',
    'charlotte perkins gilman', 'george macdonald', 'd. h. lawrence',
    'lucy maud montgomery', 'carlo collodi', 'neil gaiman',
    'george r. r. martin', 'rick riordan', 'j. k. rowling',
    'bram stoker', 'edith nesbit', 'l. frank baum',
    'richard adams', 'roald dahl', 'c. s. lewis', 'c.s. lewis',
    'stephen king', 'astrid lindgren', 'bill willingham',
    'mary shelley', 'niccolò machiavelli', 'michael ende',
    'charles kingsley', 'andrew lang', 'dr. seuss', 'lewis carroll',
    'edgar allan poe', 'nathaniel hawthorne', 'mark twain',
    'antoine de saint-exupéry', 'christopher paolini', 'j.r.r. tolkien',
    'mercedes lackey', 'piers anthony', 'mary pope osborne',
    'daisy meadows', 'erin hunter', 'paul theroux', 'harold macgrath',
    'charles de lint', 'james rollins', 'e. b. white', 'e.b. white',
    'philip pullman', 'terry pratchett', 'margaret atwood', 'george orwell',
    'frank herbert', 'isaac asimov', 'robert jordan', 'anne mccaffrey',
    'brandon sanderson', 'sarah j. maas', 'cassandra clare', 'ray bradbury',
    'hans christian andersen', 'william shakespeare', 'franz kafka',
    'iain banks', 'arthur c. clarke', 'lois mcmaster bujold',
    'susanna clarke', 'kelly link', 'jane yolen',
    # Extra western authors found via clustering
    'caroline peckham', 'neal stephenson',
    'sarah a. parker', 'samantha shannon', 'olivia blake',
    'kazuo ishiguro', 'truman capote', 'shel silverstein',
    'laura dave', 'gabrielle zevin', 'marissa meyer', 'pierce brown',
    'naomi novik', 'diana wynne jones', 'raymond e. feist',
    'lafcadio hearn', 'ruth r. carter',
    # Academic / non-fiction authors found in afrofuturism tag
    'andré m. carrington', 'achille mbembe', 'alex zamalin',
    'isiah lavender iii', 'rasheedah phillips', 'tavia nyong\'o',
    'constance backhouse', 'erik steinskog', 'sandra jackson',
    'esther l. jones', 'sami schalk', 'shelley streeby',
    'annalisa oboe', 'philip butler', 'eugen bacon',
    'reynaldo anderson', 'martin johnson', 'warren shapiro',
    'tressie mcmillan cottom', 'ibram x. kendi', 'colin kaepernick',
    'peter linebaugh', 'édouard glissant', 'duewa m. frazier',
    'kevin m. strait',
}
 
NOISE_SUBJECTS = {'tarzan', 'colonialism', 'imperialism', 'safari', 'big game hunting', 'missionaries'}
 
COMIC_SUBJECTS = {'graphic novel', 'graphic novels', 'comics', 'comic book', 'comic strip', 'sequential art'}
 
NONFICTION_SUBJECTS = {
    'criticism', 'critical essays', 'literary criticism',
    'history and criticism', 'congresses', 'study and teaching',
    'nonfiction', 'biography', 'autobiography',
    'education', 'philosophy', 'sociology', 'political science',
    'essays', 'handbooks', 'reference',
    'cross-cultural studies', 'mass media', 'self-perception',
    'creative ability in children', 'imagination in children',
}
 
NONFICTION_TITLE_SIGNALS = {
    'biography', 'autobiography', 'interviews', 'conversations with',
    'art of ', 'making of', 'the life of', 'history of',
    'study guide', 'essays on', 'criticism', 'scholarly',
    'coloring book', 'colouring book', 'activity book',
    # Added: academic series/publisher signals
    'routledge research', 'palgrave studies', 'new suns: race',
    'american studies now', 'black studies and critical thinking',
    'black literary and cultural expressions', 'osgoode society',
    'literary prehistory', 'theory & practice', 'and other essays',
    'a legal history', 'community history', 'introduction to afrofuturism',
    'partible paternity', 'anthropological theory',
}
 
ORG_SIGNALS = ['publishing llc', 'publishing inc', 'enterprises', 'museum',
               'institute', 'university', 'library of congress', 'disney', 'various']
 
TITLE_NOISE = [
    "Art of Ruth E. Carter", "Conversations with Octavia Butler",
    "Bloodchildren: Stories by the Octavia E. Butler Scholars",
    "Strange Matings: Science Fiction, Feminism, African American Voices",
    "Positive Obsession: The Life and Times of Octavia E. Butler",
    "Why Wakanda Matters", "MCU: The Reign of Marvel Studios",
    "Star Child: A Biographical Constellation", "NOT A BOOK",
    "Charlotte's Web",
    # Added: specific academic titles found in dataset
    "Speculative Blackness: The Future of Race in Science Fiction",
    "Black Quantum Futurism: Theory & Practice (Vol. 1)",
    "Afrofuturism Rising: The Literary Prehistory of a Movement",
    "Black Utopia: The History of an Idea from Black Nationalism to Afrofuturism",
    "Afrofuturism: A History of Black Futures",
    "Critique of Black Reason",
    "Black Utopias: Speculative Life and the Music of Other Worlds",
    "Black Apocalypse: Afrofuturism at the End of the World",
    "Colour-Coded: A Legal History of Racism in Canada, 1900-1950",
    "The Black Imagination: Science Fiction, Futurism and the Speculative",
    "Four Hundred Souls: A Community History of African America, 1619-2019",
    "Our History Has Always Been Contraband: In Defense of Black Studies",
    "Bodyminds Reimagined: (Dis)ability, Race, and Gender in Black Women's Speculative Fiction",
    "Medicine and Ethics in Black Women's Speculative Fiction",
    "Imagining the Future of Climate Change",
    "Black Atlantic Speculative Fictions",
    "Recharting the Black Atlantic",
    "Critical Black Futures: Speculative Theories and Explorations",
    "Afro-Centered Futurisms in Our Speculative Fiction",
    "The Black Speculative Arts Movement",
    "Introduction to Afrofuturism",
    "Afrofuturism and Black Sound Studies",
    "Literary Afrofuturism in the Twenty-First Century",
    "The Many-Headed Hydra: Sailors, Slaves, Commoners",
    "Caribbean Discourse: Selected Essays",
    "Thick: And Other Essays",
    "Partible paternity and anthropological theory",
    "Art And Scientific Thought; Historical Studies Towards A Modern Revision Of Their Antagonism",
    "New History of the DC Universe: The Dakota Incident",
]
 
JUNK_SIGNALS = [
    'tagebuch', 'notizbuch', 'malbuch', 'planer', 'kalender',
    'coloring', 'colouring', 'color by number', 'weight tracker',
    'expense tracker', 'password log', 'music sheet', 'genkoyoushi',
    'house sitting', 'self care', 'reading list book',
    'choreograph', 'dancing for', 'folk dance', 'cookbook', 'recipes',
    'guitar solo', 'diabetes', 'trivia', 'food fantasy fgb',
    'interview', 'biograpghical', 'marvel studios', 'marvel',
    'coloring book',
]
print("Filter sets defined")

Filter sets defined


## 3. Filter & Merge

In [3]:
def ol_filter(row):
    author        = str(row.get('author', '')).lower().strip()
    subjects_text = ' '.join(str(s) for s in (row.get('subjects') or [])).lower()
    if author in WESTERN_NOISE_AUTHORS: return False
    if any(n in subjects_text for n in NOISE_SUBJECTS): return False
    if any(c in subjects_text for c in COMIC_SUBJECTS): return False
    if any(n in subjects_text for n in NONFICTION_SUBJECTS): return False
    return True

def gr_filter(row):
    author     = str(row.get('author', '')).lower().strip()
    title_text = str(row.get('title', '')).lower()
    if author in WESTERN_NOISE_AUTHORS: return False
    if any(n in title_text for n in NONFICTION_TITLE_SIGNALS): return False
    return True

ol_clean = ol_raw[ol_raw.apply(ol_filter, axis=1)].copy().reset_index(drop=True)
ol_clean = ol_clean[ol_clean['author'].str.strip() != ''].reset_index(drop=True)
gr_clean = gr_raw[gr_raw.apply(gr_filter, axis=1)].copy().reset_index(drop=True)
print(f"OL: {len(ol_raw):,} -> {len(ol_clean):,}")
print(f"GR: {len(gr_raw):,} -> {len(gr_clean):,}")

OL: 4,357 -> 2,229
GR: 2,129 -> 2,045


In [4]:
def norm_title(t):
    t = str(t).lower().strip()
    t = re.sub(r'\s*[\(\[].*?[\)\]]\s*', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def norm_author(a): return str(a).lower().strip()

ol_unified = pd.DataFrame({
    'title': ol_clean['title'].str.strip(), 'author': ol_clean['author'].fillna(''),
    'description': ol_clean['description'].fillna(''), 'year_published': ol_clean['year_published'],
    'cover_url': ol_clean['cover_url'].fillna(''), 'avg_rating': ol_clean['avg_rating'],
    'num_ratings': ol_clean['num_ratings'],
    'subjects': ol_clean['subjects'].apply(lambda x: x if isinstance(x, list) else []),
    'source': 'open_library', 'source_url': ol_clean['source_url'], 'source_tag': ol_clean['source_tag'],
    'title_norm': ol_clean['title'].apply(norm_title), 'author_norm': ol_clean['author'].fillna('').apply(norm_author),
})
gr_unified = pd.DataFrame({
    'title': gr_clean['title'].str.strip(), 'author': gr_clean['author'].fillna(''),
    'description': gr_clean['description'].fillna(''), 'year_published': gr_clean['year_published'],
    'cover_url': gr_clean['cover_url'].fillna(''), 'avg_rating': gr_clean['avg_rating'],
    'num_ratings': gr_clean['num_ratings'], 'subjects': [[] for _ in range(len(gr_clean))],
    'source': 'goodreads', 'source_url': gr_clean['goodreads_url'], 'source_tag': gr_clean['shelf'],
    'title_norm': gr_clean['title'].apply(norm_title), 'author_norm': gr_clean['author'].fillna('').apply(norm_author),
})
gr_lookup = {(r['title_norm'], r['author_norm']): r for _, r in gr_unified.iterrows()}
enriched_ol = []
for _, row in ol_unified.iterrows():
    gr_match = gr_lookup.get((row['title_norm'], row['author_norm']))
    if gr_match is not None:
        row = row.copy()
        if gr_match['description']: row['description'] = gr_match['description']
        if gr_match['cover_url']: row['cover_url'] = gr_match['cover_url']
        if pd.notna(gr_match['avg_rating']): row['avg_rating'] = gr_match['avg_rating']
        if pd.notna(gr_match['num_ratings']): row['num_ratings'] = gr_match['num_ratings']
    enriched_ol.append(row)
ol_enriched = pd.DataFrame(enriched_ol)
gr_keys   = set(gr_lookup.keys())
ol_only   = ol_enriched[~ol_enriched.apply(lambda r: (r['title_norm'], r['author_norm']) in gr_keys, axis=1)]
merged    = pd.concat([gr_unified, ol_only], ignore_index=True)
merged    = merged.drop(columns=['title_norm', 'author_norm'])
print(f"Merged: {len(merged):,} books")

Merged: 4,175 books


## 4. Description Enrichment

In [5]:
def fetch_google_books_description(title, author=''):
    if not GOOGLE_BOOKS_API_KEY: return None
    params = {'q': f'{title} {author}'.strip(), 'maxResults': 5, 'key': GOOGLE_BOOKS_API_KEY,
              'fields': 'items(volumeInfo(title,authors,description,publishedDate,imageLinks))'}
    try:
        resp = requests.get('https://www.googleapis.com/books/v1/volumes', params=params, timeout=10)
        resp.raise_for_status()
        for item in resp.json().get('items', []):
            info = item.get('volumeInfo', {})
            desc = info.get('description', '')
            if desc:
                try:
                    if detect(desc) == 'en':
                        return {'description': desc, 'cover_url': info.get('imageLinks', {}).get('thumbnail', '')}
                except: continue
        return None
    except: return None

def fetch_ol_description(ol_url):
    key = ol_url.replace('https://openlibrary.org', '')
    if not key.startswith('/works/'): return None
    try:
        r = requests.get(f'https://openlibrary.org{key}.json', timeout=10)
        desc = r.json().get('description', '')
        if isinstance(desc, dict): desc = desc.get('value', '')
        return str(desc).strip() if desc and len(str(desc).strip()) > 20 else None
    except: return None

gb_results = json.load(open(GB_CKPT)) if GB_CKPT.exists() else {}
needs_desc = merged[merged['description'].apply(lambda x: not bool(str(x).strip()) or str(x).strip() == 'nan')]
remaining  = needs_desc[~needs_desc['source_url'].isin(gb_results.keys())]
print(f"Google Books: {len(gb_results):,} done | {len(remaining):,} remaining")
stats = {'filled': 0, 'not_found': 0}
for i, (idx, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining)), 1):
    result = fetch_google_books_description(row['title'], row['author'])
    gb_results[row['source_url']] = result if result else {}
    stats['filled' if result else 'not_found'] += 1
    if i % 100 == 0: json.dump(gb_results, open(GB_CKPT, 'w'))
    time.sleep(0.5)
json.dump(gb_results, open(GB_CKPT, 'w'))
for idx, row in merged.iterrows():
    gb = gb_results.get(row['source_url'], {})
    if gb.get('description') and not bool(str(row['description']).strip()):
        merged.at[idx, 'description'] = gb['description']
    if gb.get('cover_url') and not bool(str(row['cover_url']).strip()):
        merged.at[idx, 'cover_url'] = gb['cover_url']
print(f"Filled: {stats['filled']:,} | Not found: {stats['not_found']:,}")

Google Books: 2,288 done | 0 remaining


0it [00:00, ?it/s]

Filled: 0 | Not found: 0


## 5. Final Cleaning & Save

In [6]:
def is_junk(row):
    has_desc = bool(str(row.get('description', '')).strip()) and str(row.get('description', '')).strip() != 'nan'
    has_ratings = pd.notna(row.get('num_ratings')) and row.get('num_ratings', 0) > 0
    if has_desc or has_ratings: return False
    return any(s in str(row.get('title', '')).lower() for s in JUNK_SIGNALS)

def is_english(text):
    try: return detect(str(text)) == 'en'
    except: return False

def has_english_title(title):
    try: return detect(str(title)) == 'en'
    except: return True

def clean_title(title):
    title = re.sub(r'\s*[\(\[](hardcover|paperback|kindle edition|ebook|mass market paperback|audio cd|audiobook|unknown binding|library binding|board book|large print)[\)\]]', '', title, flags=re.IGNORECASE)
    return title.strip()

merged = merged[~merged.apply(is_junk, axis=1)].reset_index(drop=True)
for idx, row in merged.iterrows():
    if bool(str(row['description']).strip()) and str(row['description']).strip() != 'nan': continue
    subj = [str(s).strip() for s in (row['subjects'] if isinstance(row['subjects'], list) else []) if len(str(s).strip()) > 2][:10]
    if subj: merged.at[idx, 'description'] = 'A work of fantasy fiction involving: ' + ', '.join(subj) + '.'
merged = merged[~merged['title'].isin(TITLE_NOISE)].reset_index(drop=True)
merged = merged[~merged['title'].str.contains('coloring book', case=False, na=False)].reset_index(drop=True)
merged = merged[~merged['author'].str.lower().str.strip().apply(lambda a: any(s in a for s in ORG_SIGNALS))].reset_index(drop=True)
fallback_mask = merged['description'].str.contains('A work of fantasy fiction involving', na=False)
non_english   = ~merged['title'].apply(has_english_title)
merged = merged[~(fallback_mask & non_english)].reset_index(drop=True)
merged = merged[merged['description'].apply(is_english)].reset_index(drop=True)
merged['title'] = merged['title'].apply(clean_title)
before = len(merged)
merged['_tn'] = merged['title'].str.lower().str.strip()
merged['_an'] = merged['author'].str.lower().str.strip()
merged['_sr'] = merged['source'].map({'goodreads': 0, 'open_library': 1})
merged = merged.sort_values('_sr').drop_duplicates(subset=['_tn', '_an'], keep='first').drop(columns=['_tn', '_an', '_sr']).reset_index(drop=True)
print(f"Duplicates removed: {before - len(merged):,}")
merged['year_published'] = pd.to_numeric(merged['year_published'], errors='coerce').fillna(0).astype(int).replace(0, None)
merged['num_ratings']    = pd.to_numeric(merged['num_ratings'], errors='coerce').fillna(0).astype(int)
merged['avg_rating']     = merged['avg_rating'].round(2)
merged['subjects']       = merged['subjects'].apply(lambda x: x if isinstance(x, list) else [])
for col in ['title', 'author', 'description', 'cover_url', 'source_url', 'source_tag']:
    merged[col] = merged[col].str.strip()
merged.to_json(MERGED_OUT, orient='records', indent=2, force_ascii=False)
merged.to_csv(str(MERGED_OUT).replace('.json', '.csv'), index=False)
print(f"\nDone! {len(merged):,} books saved to {MERGED_OUT}")

Duplicates removed: 55

Done! 3,247 books saved to C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Clean\merged_non_western_fantasy.json


## Conclusions

- Goodreads wins on overlaps — higher quality descriptions and ratings
- ~700 books removed: coloring books, non-English, western authors, nonfiction
- HDBSCAN clustering in NB04 acted as a data quality check, revealing hidden noise
- English filter ensures TF-IDF works on meaningful vocabulary